In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import gc
import os
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. CONFIGURATION & FILE SETUP
# ==========================================
train_path = "/kaggle/input/competitions/ts-forecasting/train.parquet"
TARGET = "y_target"
FORECAST_WINDOWS = [1, 3, 10, 25]

LOG_CSV = "cv_random_search_results.csv"
BEST_CSV = "best_cv_params_per_horizon.csv"

# Safely initialize the master log CSV
if not os.path.exists(LOG_CSV):
    header = pd.DataFrame(columns=[
        'horizon', 'iteration', 'n_folds', 'oof_wrmse', 'avg_best_trees',
        'learning_rate', 'n_estimators', 'num_leaves', 'min_child_samples', 
        'feature_fraction', 'bagging_fraction', 'bagging_freq'
    ])
    header.to_csv(LOG_CSV, index=False)
    print(f"📁 Created new master log: {LOG_CSV}")
else:
    print(f"📁 Found existing master log {LOG_CSV}, appending to it.")

# Safely load previous best scores if resuming
best_scores = {h: -1.0 for h in FORECAST_WINDOWS}
best_params_dict = {h: {} for h in FORECAST_WINDOWS}

if os.path.exists(BEST_CSV):
    print(f"📁 Found existing best configurations in {BEST_CSV}. Loading high scores into memory...")
    best_df = pd.read_csv(BEST_CSV)
    for _, row in best_df.iterrows():
        hz = int(row['horizon'])
        best_scores[hz] = float(row['oof_wrmse'])
        best_params_dict[hz] = row.to_dict()
    print("📈 Current High Scores to beat:")
    for hz, score in best_scores.items():
        if score > -1.0:
            print(f"   - Horizon {hz}: {score:.5f}")
else:
    print(f"📁 No previous bests found. Starting fresh.")

# ==========================================
# 1. METRICS & TRUE RANDOM SAMPLER
# ==========================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

def get_random_params():
    """Generates fresh parameters independently every time it is called."""
    return {
        'n_folds': int(np.random.randint(6, 12)),
        'lgb_params': {
            'objective': 'regression',
            'metric': 'rmse',
            'learning_rate': float(np.exp(np.random.uniform(np.log(0.001), np.log(0.1)))),
            'n_estimators': int(np.random.randint(500, 2000)),
            'num_leaves': int(np.random.randint(32, 256)),
            'min_child_samples': int(np.random.randint(10, 300)),
            'bagging_freq': int(np.random.randint(1, 15)),
            'feature_fraction': float(np.random.uniform(0.4, 1.0)),
            'bagging_fraction': float(np.random.uniform(0.4, 1.0)),
            'verbosity': -1,
            'n_jobs': -1,
            'random_state': 42 
        }
    }

# ==========================================
# 2. DATA PREP & CACHING
# ==========================================
print("\nLoading train.parquet and caching horizons into memory...")
train_full = pd.read_parquet(train_path)

horizon_cache = {}

for horizon in FORECAST_WINDOWS:
    tr_df = train_full[train_full['horizon'] == horizon].reset_index(drop=True)

    if len(tr_df) == 0:
        continue

    # Identify features
    drop_cols = [TARGET, 'ts_index', 'horizon', 'id']
    features = sorted([c for c in tr_df.columns if c not in drop_cols])

    X = tr_df[features].copy()
    y = tr_df[TARGET].values
    weights = tr_df['weight'].values if 'weight' in tr_df.columns else np.ones_like(y)

    cat_cols = X.select_dtypes(exclude=['number', 'bool']).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].astype('category')
        
    horizon_cache[horizon] = {'X': X, 'y': y, 'weights': weights}
    print(f"✅ Horizon {horizon} prepped: {len(X)} rows")

del train_full
gc.collect()

# ==========================================
# 3. CONTINUOUS OOF RANDOM SEARCH LOOP
# ==========================================
print("\n🚀 Starting continuous OOF Random Search. Kill the script manually when satisfied.")

iteration = 1

while True:
    print(f"\n{'='*55}\n 🔄 GLOBAL ITERATION: {iteration}\n{'='*55}")
    
    for horizon, data in horizon_cache.items():
        X = data['X']
        y = data['y']
        weights = data['weights']
        
        # Fresh random parameters for this specific horizon
        rand_config = get_random_params()
        n_folds = rand_config['n_folds']
        lgb_cfg = rand_config['lgb_params']
        
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
        oof_preds = np.zeros(len(X))
        fold_best_iters = []
        
        for fold, (trn_idx, val_idx) in enumerate(kf.split(X)):
            X_trn, y_trn = X.iloc[trn_idx], y[trn_idx]
            X_val, y_val = X.iloc[val_idx], y[val_idx]
            
            w_trn, w_val = weights[trn_idx], weights[val_idx]

            model = lgb.LGBMRegressor(**lgb_cfg)
            model.fit(
                X_trn, y_trn,
                sample_weight=w_trn,
                eval_set=[(X_val, y_val)],
                eval_sample_weight=[w_val],
                callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
            )
            
            oof_preds[val_idx] = model.predict(X_val)
            fold_best_iters.append(model.best_iteration_)
            
        horizon_total_score = weighted_rmse_score(y, oof_preds, weights)
        avg_trees = int(np.mean(fold_best_iters))
        
        # Package run details
        run_dict = {
            'horizon': horizon,
            'iteration': iteration,
            'n_folds': n_folds,
            'oof_wrmse': horizon_total_score,
            'avg_best_trees': avg_trees,
            'learning_rate': lgb_cfg['learning_rate'],
            'n_estimators': lgb_cfg['n_estimators'],
            'num_leaves': lgb_cfg['num_leaves'],
            'min_child_samples': lgb_cfg['min_child_samples'],
            'feature_fraction': lgb_cfg['feature_fraction'],
            'bagging_fraction': lgb_cfg['bagging_fraction'],
            'bagging_freq': lgb_cfg['bagging_freq']
        }
        
        # Save to Master Log
        pd.DataFrame([run_dict]).to_csv(LOG_CSV, mode='a', header=False, index=False)
        
        # Track and Save Best Models
        is_best = ""
        if horizon_total_score > best_scores[horizon]:
            best_scores[horizon] = horizon_total_score
            best_params_dict[horizon] = run_dict
            is_best = "🌟 NEW BEST!"
            
            # Immediately overwrite the Best CSV with updated high scores
            current_bests = [best_params_dict[h] for h in FORECAST_WINDOWS if best_params_dict[h]]
            pd.DataFrame(current_bests).to_csv(BEST_CSV, index=False)
            
        print(f"Hz {horizon:<2} | {n_folds:2d} Folds | OOF WRMSE: {horizon_total_score:.5f} | Avg Trees: {avg_trees:<4} {is_best}")
        
    iteration += 1

Loading train.parquet and applying consistent 80/20 splits...
✅ Horizon 1 prepped: Train=1115722 rows | Holdout=278931 rows
✅ Horizon 3 prepped: Train=1108652 rows | Holdout=277164 rows
✅ Horizon 10 prepped: Train=1069788 rows | Holdout=267448 rows
✅ Horizon 25 prepped: Train=975767 rows | Holdout=243942 rows

🚀 Starting continuous round-robin search. Kill the script manually when satisfied.

 🔄 GLOBAL ITERATION: 1


KeyboardInterrupt: 